In [1]:
# All paths in this notebook are relative to the repository root; anchor the working directory there
import os
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'

# Corridor Transit Profiles — Calibrated Survey vs Ticketing-Based

Compares the two transit profiles of potential movements along the LRT corridor, link by link and direction by direction (three-hour totals, not loads):

| Profile | Bus layer | Rail layer | Frame |
|---|---|---|---|
| **Calibrated survey** (file names still say "hybrid") — `Corridor_flow_profile_survey_2022.ipynb` | survey Public Bus + Matronit, destination pattern blended with RavKav × OnBoard, volumes from RavKav per origin × segment where ticketing coverage is credible, grown to 2022 where not | survey rail, door-to-door, × 0.793 | residents, doorstep origins; 2022 |
| **Ticketing-based** — `Transit_complete_matrix.ipynb` / `Vintage_alignment_2022.ipynb` | RavKav journey volumes × OnBoard destinations, May 2022 | 2019 smartcard station-to-station × 0.793 | all riders, boarding-stop origins; 2022 |

Transit is **bus + rail** in both sets, so the comparison is like-for-like; the survey's taxi-type layer is shown separately and never added to transit. Both use the same 18-area line sequence and the same link-assignment rule (each OD pair with both ends on the line is added to every link between them). The comparison:

1. link flows of the two profiles and their components (bus, rail / train; taxi-type alongside), per direction;
2. the intermediate steps of the calibration — raw survey bus (2018), the all-RavKav variant — to show what each step does to the profile;
3. which area pairs drive the difference on the busiest links;
4. transit share along the line for both matrix sets (transit ÷ total profile);
5. ticketing coverage in the guarded superzones split into local and inter-superzone movements, with the segments the segmented rule guards.

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLUE, ORANGE, AQUA, PURPLE, INK, INK2, MUTED, GRID, AXIS = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/ths2017/three_mode_2022'
sub_key = pd.read_excel('Input/Submatrix_tazs.xlsx'); area_of = sub_key.set_index('TAZ')['AggAreaCode']; AREA = sorted(area_of.unique())
legend = pd.read_csv('Output/ths2017/study_taz/submatrices/area_legend.csv').set_index('AggAreaCode'); names = legend['AggAreaName']
ref = pd.read_csv('Output/transit/corridor_link_flows_transit_2022.csv'); SEQ = list(ref['from_code']) + [int(ref['to_code'].iloc[-1])]

def load(path):
    m = pd.read_csv(path, index_col=0); m.index = m.index.astype(int); m.columns = m.columns.astype(int); return m
def taz_to_area(m):
    m = m.copy(); m.index.name, m.columns.name = 'o', 'd'
    long = m.stack().reset_index(); long.columns = ['o', 'd', 'v']
    long['O'] = long['o'].map(area_of); long['D'] = long['d'].map(area_of)
    return long.dropna(subset=['O', 'D']).groupby(['O', 'D'])['v'].sum().unstack().reindex(index=AREA, columns=AREA, fill_value=0).fillna(0)
A = lambda m: m.reindex(index=AREA, columns=AREA).fillna(0)

# hybrid (calibrated survey) components, 2022
H_bus = taz_to_area(load(f'{OUT}/bus_2022_taz.csv'))                     # scheduled bus only (taxi-type is its own layer)
H_taxi = taz_to_area(load(f'{OUT}/taxi_2022_taz.csv'))
H_rail = A(load(f'{OUT}/rail_2022_area.csv'))
H_total = A(load(f'{OUT}/car_2022_area.csv')) + A(load(f'{OUT}/bus_2022_area.csv')) + A(load(f'{OUT}/taxi_2022_area.csv')) + H_rail
# ticketing-based components, 2022
T_bus = taz_to_area(load('Output/bus/bus_od_taz_avg.csv'))            # ticketing reference: RavKav journeys on RavKav's own alightings (revised 23 Sep 2026, §6ag); the OnBoard-patterned matrix of step 9 is no longer the reference
T_train = A(load('Output/train/train_od_area.csv')) * (54.7 / 69.0)
T_total = A(load('Output/transit/all_adjusted_area_2022.csv'))
# calibration steps (2018 survey bus, all-RavKav variant), for the "what each step does" view
S_bus18 = A(load('Output/ths2017/two_mode/bus_survey_sz.csv').iloc[0:0]) if False else taz_to_area(load('Output/ths2017/two_mode/bus_survey_taz.csv'))
R_bus = taz_to_area(load('Output/ths2017/two_mode/bus_calibrated_all_ravkav_taz.csv'))
print(f"sub-area totals — hybrid: bus {H_bus.values.sum():,.0f}, taxi-type {H_taxi.values.sum():,.0f}, rail {H_rail.values.sum():,.0f} | "
      f"ticketing: bus {T_bus.values.sum():,.0f}, train {T_train.values.sum():,.0f} | survey bus 2018 {S_bus18.values.sum():,.0f}, all-RavKav variant {R_bus.values.sum():,.0f}")

sub-area totals — hybrid: bus 25,858, taxi-type 676, rail 104 | ticketing: bus 23,995, train 763 | survey bus 2018 25,568, all-RavKav variant 23,701


In [3]:
def link_flows(m, seq=SEQ):
    sub = m.reindex(index=seq, columns=seq).fillna(0).values; n = len(seq)
    f, b = np.zeros(n - 1), np.zeros(n - 1)
    for i in range(n):
        for j in range(n):
            if i < j: f[i:j] += sub[i, j]
            elif i > j: b[j:i] += sub[i, j]
    return f, b
COMP = {'hybrid: bus (calibrated survey)': H_bus, 'hybrid: taxi-type': H_taxi, 'hybrid: rail (survey)': H_rail,
        'ticketing: bus (RavKav × OnBoard)': T_bus, 'ticketing: train (station)': T_train,
        'step: survey bus 2018 (raw)': S_bus18, 'step: bus, all-RavKav volumes': R_bus, 'total: hybrid set': H_total, 'total: ticketing set': T_total}
LF = {k: link_flows(v) for k, v in COMP.items()}
links = [f'{names[a]} – {names[b]}' for a, b in zip(SEQ[:-1], SEQ[1:])]
tbl = pd.DataFrame({'link': links})
for d, di in [('1→23', 0), ('23→1', 1)]:
    tbl[f'hybrid transit {d}'] = LF['hybrid: bus (calibrated survey)'][di] + LF['hybrid: rail (survey)'][di]          # bus + rail, like-for-like with ticketing bus + train
    tbl[f'hybrid taxi-type {d}'] = LF['hybrid: taxi-type'][di]
    tbl[f'ticketing transit {d}'] = LF['ticketing: bus (RavKav × OnBoard)'][di] + LF['ticketing: train (station)'][di]
    tbl[f'diff {d}'] = tbl[f'hybrid transit {d}'] - tbl[f'ticketing transit {d}']
    tbl[f'ratio {d}'] = tbl[f'hybrid transit {d}'] / tbl[f'ticketing transit {d}'].replace(0, np.nan)
    tbl[f'transit share hybrid {d}'] = tbl[f'hybrid transit {d}'] / LF['total: hybrid set'][di]
    tbl[f'transit share ticketing {d}'] = tbl[f'ticketing transit {d}'] / LF['total: ticketing set'][di]
assert np.allclose(tbl['ticketing transit 1→23'].round(0), ref['flow_dir_1_to_23']) and np.allclose(tbl['ticketing transit 23→1'].round(0), ref['flow_dir_23_to_1']), "ticketing profile must reproduce the saved one"
tbl.to_csv(f'{OUT}/corridor_profile_hybrid_vs_ticketing.csv', index=False, float_format='%.3f')
comp = pd.DataFrame({'link': links, **{f'{k} {d}': LF[k][di] for k in COMP for d, di in [('1→23', 0), ('23→1', 1)]}})
comp.to_csv(f'{OUT}/corridor_profile_components.csv', index=False, float_format='%.1f')
print("link flows, transit, both profiles (trips 6:00–9:00):")
tbl[['link', 'hybrid transit 1→23', 'ticketing transit 1→23', 'ratio 1→23', 'hybrid taxi-type 1→23', 'hybrid transit 23→1', 'ticketing transit 23→1', 'ratio 23→1', 'hybrid taxi-type 23→1']].round(2)

link flows, transit, both profiles (trips 6:00–9:00):


,link,hybrid transit 1→23,ticketing transit 1→23,ratio 1→23,hybrid taxi-type 1→23,hybrid transit 23→1,ticketing transit 23→1,ratio 23→1,hybrid taxi-type 23→1
0,TiratCarmel – Matam,769.30,776.07,0.99,0.0,319.10,191.83,1.66,0.00
1,Matam – Neot Peres,836.69,1047.23,0.80,0.0,966.27,419.40,2.30,0.00
2,Neot Peres – Neve David,772.63,1037.18,0.74,0.0,1497.31,873.96,1.71,96.02
3,Neve David – Ein Hayam,1267.88,1631.89,0.78,0.0,1330.54,1628.83,0.82,39.52
4,Ein Hayam – Bat Galim,1847.22,1853.40,1.00,0.0,1000.76,1648.08,0.61,0.00
5,Bat Galim – Kiryat Eliezer,1652.52,1656.26,1.00,0.0,1119.74,2584.27,0.43,126.29
6,Kiryat Eliezer – Hamoshava,1629.10,1554.34,1.05,0.0,772.01,2397.50,0.32,131.65
7,Hamoshava – Lower City,1768.15,1461.26,1.21,0.0,779.21,2325.80,0.34,177.95
8,Lower City – Hadar Carmel,1238.15,1174.72,1.05,0.0,1093.22,2736.12,0.40,177.95
9,Hadar Carmel – Namal,884.60,724.68,1.22,0.0,1178.19,2651.99,0.44,123.23


In [4]:
sums = {}
for d, di in [('1→23', 0), ('23→1', 1)]:
    sums[d] = {'hybrid transit link-trips': tbl[f'hybrid transit {d}'].sum(), 'ticketing transit link-trips': tbl[f'ticketing transit {d}'].sum(),
               'hybrid peak': tbl[f'hybrid transit {d}'].max(), 'ticketing peak': tbl[f'ticketing transit {d}'].max(),
               'hybrid peak link': links[tbl[f'hybrid transit {d}'].idxmax()], 'ticketing peak link': links[tbl[f'ticketing transit {d}'].idxmax()],
               'Haifa segment (Tirat Carmel–Hadar) hybrid/ticketing': tbl[f'hybrid transit {d}'][:9].sum() / tbl[f'ticketing transit {d}'][:9].sum(),
               'Krayot–Nazareth segment (Namal–Nazareth) hybrid/ticketing': tbl[f'hybrid transit {d}'][10:].sum() / tbl[f'ticketing transit {d}'][10:].sum()}
pd.DataFrame(sums).T

,hybrid transit link-trips,ticketing transit link-trips,hybrid peak,ticketing peak,hybrid peak link,ticketing peak link,Haifa segment (Tirat Carmel–Hadar) hybrid/ticketing,Krayot–Nazareth segment (Namal–Nazareth) hybrid/ticketing
1→23,14506.184363,14402.149329,1847.224622,1853.400225,Ein Hayam – Bat Galim,Ein Hayam – Bat Galim,0.966315,1.238905
23→1,14394.900657,29449.88632,1497.310908,2736.11781,Neot Peres – Neve David,Lower City – Hadar Carmel,0.599641,0.361784


## What each calibration step does to the profile

Raw survey bus (2018) → volumes from RavKav everywhere (all-RavKav variant) → coverage-guarded volumes + 2022 growth (the hybrid bus), against the ticketing bus. Taxi-type and rail / train are shown separately.

In [5]:
fig, axes = plt.subplots(2, 1, figsize=(15, 11), facecolor='white', sharex=True)
x = np.arange(len(SEQ))
for ax, (d, di) in zip(axes, [('1→23', 0), ('23→1', 1)]):
    for k, color, ls, lw in [('step: survey bus 2018 (raw)', MUTED, ':', 1.6), ('step: bus, all-RavKav volumes', PURPLE, '--', 1.4),
                             ('hybrid: bus (calibrated survey)', BLUE, '-', 2.2), ('ticketing: bus (RavKav × OnBoard)', ORANGE, '-', 2.2)]:
        ax.stairs(LF[k][di], x, color=color, linestyle=ls, linewidth=lw, label=k)
    ax.stairs(LF['hybrid: bus (calibrated survey)'][di] + LF['hybrid: rail (survey)'][di], x, color=BLUE, linewidth=1, linestyle='-.', alpha=0.7, label='calibrated-survey transit (bus + rail)')
    ax.stairs(LF['hybrid: taxi-type'][di], x, color=AQUA, linewidth=1.2, linestyle=':', label='taxi-type (separate layer, not in transit)')
    ax.stairs(LF['ticketing: bus (RavKav × OnBoard)'][di] + LF['ticketing: train (station)'][di], x, color=ORANGE, linewidth=1, linestyle='-.', alpha=0.7, label='ticketing transit incl. train')
    ax.set_xlim(0, len(SEQ) - 1); ax.grid(True, color=GRID, linewidth=0.6, axis='y'); ax.set_axisbelow(True)
    for s in ax.spines.values(): s.set_color(AXIS)
    ax.tick_params(colors=INK2); ax.set_ylabel('trips crossing the link, 06:00–09:00 (3-hour totals)', color=INK2)
    ax.set_title(f'Direction {d}  ({names[SEQ[0]] if d == "1→23" else names[SEQ[-1]]} → {names[SEQ[-1]] if d == "1→23" else names[SEQ[0]]})', color=INK, fontsize=12)
    ax.legend(frameon=False, fontsize=9, loc='upper right' if d == '1→23' else 'upper left')
axes[1].set_xticks(x); axes[1].set_xticklabels([f'{a} · {names[a]}' for a in SEQ], rotation=55, ha='right', fontsize=9, color=INK2)
fig.suptitle('Corridor bus / transit potential movements — calibrated survey vs ticketing, and the calibration steps in between (3-hour totals, not loads)', color=INK, fontsize=14, y=0.995)
plt.tight_layout(); fig.savefig('Output/figures/corridor_profile_hybrid_vs_ticketing.png', dpi=150, bbox_inches='tight', facecolor='white'); plt.show()

## Which area pairs make the difference

For each direction, the contribution of an area pair to the profile is its flow × the number of links it crosses. The table lists the pairs with the largest positive (hybrid > ticketing) and negative (ticketing > hybrid) contributions to the transit profile, and the same aggregated by origin and by destination area.

In [6]:
pos = {a: i for i, a in enumerate(SEQ)}
Hm = (H_bus + H_rail).reindex(index=SEQ, columns=SEQ).fillna(0); Tm = (T_bus + T_train).reindex(index=SEQ, columns=SEQ).fillna(0)
rows = []
for o in SEQ:
    for d in SEQ:
        if o == d: continue
        nl = abs(pos[d] - pos[o])
        rows.append({'origin': names[o], 'destination': names[d], 'direction': '1→23' if pos[d] > pos[o] else '23→1', 'links crossed': nl,
                     'hybrid trips': Hm.loc[o, d], 'ticketing trips': Tm.loc[o, d], 'diff trips': Hm.loc[o, d] - Tm.loc[o, d], 'diff link-trips': (Hm.loc[o, d] - Tm.loc[o, d]) * nl})
pairs = pd.DataFrame(rows); pairs.to_csv(f'{OUT}/corridor_profile_pair_contributions.csv', index=False, float_format='%.1f')
print("largest contributions to the difference (hybrid − ticketing), link-trips:")
print(pd.concat([pairs.sort_values('diff link-trips').head(10), pairs.sort_values('diff link-trips').tail(10)]).round(0).to_string(index=False))
by_o = pairs.groupby('origin')[['hybrid trips', 'ticketing trips', 'diff link-trips']].sum().sort_values('diff link-trips')
by_d = pairs.groupby('destination')[['hybrid trips', 'ticketing trips', 'diff link-trips']].sum().sort_values('diff link-trips')
print("\nby origin area (corridor-internal transit trips and link-trip difference):"); print(by_o.round(0).to_string())
print("\nby destination area:"); print(by_d.round(0).to_string())

largest contributions to the difference (hybrid − ticketing), link-trips:
             origin destination direction  links crossed  hybrid trips  ticketing trips  diff trips  diff link-trips
      Nazareth Area   Bat Galim      23→1             12          23.0            480.0      -457.0          -5484.0
      Nazareth Area  Neve David      23→1             14           6.0            236.0      -230.0          -3214.0
      Nazareth Area  Neot Peres      23→1             15          22.0            168.0      -146.0          -2197.0
          Hamifrats  Neve David      23→1              9           1.0            158.0      -157.0          -1411.0
          Hamifrats  Lower City      23→1              4          63.0            310.0      -248.0           -990.0
         Neve Yosef  Neve David      23→1              8           4.0            120.0      -115.0           -924.0
      Nazareth Area   Hamifrats      23→1              5          31.0            207.0      -177.0        

## Transit share along the line — both matrix sets

In [7]:
fig, ax = plt.subplots(figsize=(15, 5.5), facecolor='white')
for d, di, ls in [('1→23', 0, '-'), ('23→1', 1, '--')]:
    ax.stairs(tbl[f'transit share hybrid {d}'].values, x, color=BLUE, linestyle=ls, linewidth=2, label=f'calibrated-survey set (car + bus + taxi + rail), {d}')
    ax.stairs(tbl[f'transit share ticketing {d}'].values, x, color=ORANGE, linestyle=ls, linewidth=2, label=f'ticketing set (car / other + bus + train), {d}')
ax.set_xticks(x); ax.set_xticklabels([f'{a} · {names[a]}' for a in SEQ], rotation=55, ha='right', fontsize=9, color=INK2)
ax.set_xlim(0, len(SEQ) - 1); ax.set_ylim(0, 1); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.grid(True, color=GRID, linewidth=0.6, axis='y'); ax.set_axisbelow(True)
for s in ax.spines.values(): s.set_color(AXIS)
ax.tick_params(colors=INK2); ax.set_ylabel('transit share of link flow', color=INK2)
ax.set_title('Transit (bus + rail) share of the corridor link flow — calibrated-survey set (car + bus + taxi + rail) vs ticketing set (its total includes walk / other modes)', color=INK, fontsize=12)
ax.legend(frameon=False, fontsize=9, ncol=2)
plt.tight_layout(); fig.savefig('Output/figures/corridor_profile_transit_share.png', dpi=150, bbox_inches='tight', facecolor='white'); plt.show()
tbl[['link', 'transit share hybrid 1→23', 'transit share ticketing 1→23', 'transit share hybrid 23→1', 'transit share ticketing 23→1']].round(3)

,link,transit share hybrid 1→23,transit share ticketing 1→23,transit share hybrid 23→1,transit share ticketing 23→1
0,TiratCarmel – Matam,0.212,0.153,0.246,0.178
1,Matam – Neot Peres,0.291,0.253,0.306,0.178
2,Neot Peres – Neve David,0.274,0.264,0.385,0.274
3,Neve David – Ein Hayam,0.303,0.265,0.355,0.343
4,Ein Hayam – Bat Galim,0.303,0.274,0.329,0.421
5,Bat Galim – Kiryat Eliezer,0.277,0.216,0.184,0.341
6,Kiryat Eliezer – Hamoshava,0.263,0.219,0.176,0.398
7,Hamoshava – Lower City,0.339,0.247,0.250,0.502
8,Lower City – Hadar Carmel,0.307,0.208,0.276,0.452
9,Hadar Carmel – Namal,0.326,0.188,0.339,0.545


## Why the Nazareth end differs: ticketing coverage is a local-trip problem

The coverage guard of `THS_2017_two_mode_matrix.ipynb` compared **all** bus trips per origin superzone. Splitting them into local (inside the superzone) and inter-superzone trips shows where the ticketing shortfall actually sits.

In [8]:
KEYS_LFS = 'Input/Matrices/1270_02_09_2021_TAZ_North_keys.csv'
keys_raw = pd.read_csv('Input/taz_keys_from_shapefile.csv') if open(KEYS_LFS, 'rb').read(40).startswith(b'version https://git-lfs') else pd.read_csv(KEYS_LFS, encoding='windows-1255')
sz = keys_raw[['TAZ_NUMBER', 'SZ_NEW']].dropna().astype(int).drop_duplicates('TAZ_NUMBER').set_index('TAZ_NUMBER')['SZ_NEW']
sv_taz = load('Output/ths2017/two_mode/bus_survey_taz.csv'); rk_taz = load('Output/bus/bus_od_taz_avg.csv').reindex(index=sv_taz.index, columns=sv_taz.index).fillna(0)
def to_szm(m):
    o = m.index.map(sz); d = m.columns.map(sz); return m.groupby(o).sum().T.groupby(d).sum().T
S, R = to_szm(sv_taz), to_szm(rk_taz)
HAIFA = [12, 13, 14, 15, 16, 17, 18]
fac = pd.read_csv('Output/ths2017/two_mode/bus_calibration_factors_sz.csv').set_index('origin SZ')
seg = pd.read_csv('Output/ths2017/two_mode/bus_calibration_factors_segments.csv')
gseg = seg[seg['guarded']].groupby('origin SZ')['segment'].apply(lambda s: ' / '.join(s))
rows = []
for z in [19, 4, 23, 30, 34, 33, 37, 17, 14, 16, 9]:
    rows.append({'SZ': z, 'localities': fac.loc[z, 'main localities'], 'guarded (binary rule)': str(fac.loc[z, 'flag']).startswith('ticketing'), 'guarded segments (segmented rule)': gseg.get(z, ''),
                 'survey local': S.loc[z, z], 'RavKav local': R.loc[z, z], 'ratio local': R.loc[z, z] / S.loc[z, z],
                 'survey inter-SZ': S.loc[z].sum() - S.loc[z, z], 'RavKav inter-SZ': R.loc[z].sum() - R.loc[z, z], 'ratio inter-SZ': (R.loc[z].sum() - R.loc[z, z]) / (S.loc[z].sum() - S.loc[z, z]),
                 'survey → Haifa SZs': S.loc[z, HAIFA].sum(), 'RavKav → Haifa SZs': R.loc[z, HAIFA].sum()})
cov = pd.DataFrame(rows); cov.to_csv(f'{OUT}/corridor_profile_coverage_local_vs_intercity.csv', index=False, float_format='%.2f')
cov.round(2)

,SZ,localities,guarded (binary rule),guarded segments (segmented rule),survey local,RavKav local,ratio local,survey inter-SZ,RavKav inter-SZ,ratio inter-SZ,survey → Haifa SZs,RavKav → Haifa SZs
0,19,"נצרת, כפר כנא",True,"local / inter-SZ, other",5669.41,505.66,0.09,5914.73,2137.09,0.36,1679.92,1237.39
1,4,"שפרעם, טמרה",True,"inter-SZ, corridor-bound / inter-SZ, other",492.98,396.03,0.80,1548.31,475.97,0.31,1358.21,278.43
2,23,"דאלית אל-כרמל, מועצה מקומית עוספיא",True,"local / inter-SZ, corridor-bound",849.64,145.85,0.17,958.46,642.65,0.67,821.56,622.10
3,30,"סח'נין, עראבה",True,"local / inter-SZ, corridor-bound / inter-SZ, o...",508.43,122.43,0.24,508.07,167.07,0.33,158.07,67.88
4,34,"מעלות-תרשיחא, בית ג'ן",True,"local / inter-SZ, corridor-bound / inter-SZ, o...",1454.62,151.81,0.10,1312.91,736.15,0.56,0.00,149.08
5,33,"צפת, מועצה אזורית מרום גליל",True,local,2450.00,411.11,0.17,530.74,646.89,1.22,175.00,68.44
6,37,"בית שאן, מועצה אזורית עמק המעיינות",True,"local / inter-SZ, corridor-bound / inter-SZ, o...",665.33,30.00,0.05,1251.55,692.25,0.55,0.00,565.68
7,17,חיפה,False,,2566.83,1905.27,0.74,5275.89,5062.60,0.96,6898.41,6410.81
8,14,חיפה,False,local,1317.89,623.36,0.47,4528.66,3678.11,0.81,5397.79,3530.60
9,16,חיפה,False,local,4339.12,1913.07,0.44,4918.29,3419.93,0.70,8708.34,4885.04


## Findings (after the segmented coverage rule and the taxi split)

1. **Towards Nazareth (1 → 23) the two transit profiles agree.** With taxi-type out of the transit layer, the calibrated-survey profile sits at 0.63–1.05 × the ticketing one along the Haifa segment (Tirat Carmel → Hadar Carmel; 0.87 over the segment as a whole), the raw 2018 survey 25–40 % lower. The earlier 1.5 × was entirely the taxi-type layer, which is now the dotted line (up to 1,415 trips on Ein Hayam – Bat Galim).
2. **Towards Tirat Carmel (23 → 1) the ticketing profile is still about double** from Bat Galim to Kiryat Bialik South (ratios 0.43–0.58) and about three times at the Nazareth end (0.32–0.37). The segmented rule did **not** close this gap, and the pair table shows why it could not: the driver is journeys ticketed from the Nazareth Area (1,363 vs 430 survey-based, −9,900 link-trips), Hamifrats (782 vs 561) and Neve Yosef (741 vs 296) into Haifa's western districts (Bat Galim, Neve David, Neot Peres, Lower City). At superzone level the Nazareth superzone's corridor-bound segment is *calibrated* (ratio 0.74, so it took the ticketing volume of 1,237); the shortfall is inside the superzone-to-area allocation and in the destination frame — the survey's Nazareth → Haifa bus trips land in Haifa TAZs by employment share, ticketing's at trunk-route alighting stops in the line areas — plus the hub attribution at Hamifrats and Neve Yosef. Coverage was the wrong diagnosis for the corridor-bound market; the local market is where it holds.
3. **The coverage gap is a local-trip gap, and the rule now follows that.** In the Nazareth superzone ticketing records 8 % of the survey's *local* bus trips but 34 % of its inter-superzone trips and 74 % of its trips to the Haifa superzones; the other guarded superzones show the same shape (local ratios 0.05–0.24). The segmented rule keeps survey volumes for the local segment of 17 superzones (the seven originally guarded ones, Haifa 14 and 16 at 0.44–0.48, Tiberias, Afula, Migdal HaEmek, Nof HaGalil, Pardes Hanna, Hadera and others), and for inter-superzone segments only where those also fall below 0.5 with at least 5 sampled trips (Shefa-'Amr / Tamra both segments, Nazareth 'other', Ma'alot 'other', Beit She'an 'other', Pardes Hanna 'other') or inherit a guarded origin's decision because they hold fewer than 5 sampled trips (Sakhnin, Daliyat al-Karmel, Ma'alot and Beit She'an corridor-bound — three of them with ratios above 1 on 1–3 trips). What a sub-0.5 ratio means — missing operators, cash fares, or survey over-expansion of short bus trips — is still unresolved (review §8).
4. **Transit share of link flow** (bus + rail over car + bus + taxi + rail) is 16–26 % (1 → 23) and 14–37 % (23 → 1) in the calibrated-survey set across the Haifa segment, against 15–27 % and 18–55 % in the ticketing set, whose total also contains walk / other modes. The Krayot–Nazareth segment is where the sets disagree most: the calibrated-survey set has almost no transit heading towards Nazareth in the morning, the ticketing set 60–90 % transit heading towards Haifa — the latter is the combination of hub-attributed journeys and a thin car layer at that end of the earlier matrix.
5. **Rail is immaterial to both profiles** (survey rail: no corridor-internal trips; station-based train: 763 sub-area trips, concentrated at station areas).

## Notes

- **Two frames, not two estimates of one thing.** The ticketing profile counts every rider who boarded in a corridor area, including non-residents and journeys that start with a transfer at a hub (Hamifrats, Neve Yosef); the calibrated-survey profile counts residents from their doorstep. Where they differ most is where those frames differ most. A boarding-based product and a person-journey product should stay separate until an access / transfer allocation links them (review §3).
- **Three-hour totals.** Every link value is potential movement over 06:00–09:00; the peak values (1,661 calibrated-survey, 2,736 ticketing) are not peak-hour loads.
- **Rail.** Door-to-door survey rail is tiny inside the corridor (no corridor-internal trips at all); the station matrix puts train trips at station areas. Both are small against bus.
- **Transit share** in the ticketing set is measured against a total that includes walking and other modes, so it sits lower by construction.